# QM 640 Capstone — Data Cleaning and Integration (Version 2)

**Project:** Predicting Catastrophe Claim Severity Using Hazard, Socioeconomic, and Machine Learning Methods  
**Notebook:** `01_Data_Cleaning_v2.ipynb`  
**Unit of analysis:** U.S. county-year  
**Data sources:** NOAA Storm Events, FEMA/NFIP Flood Claims, and Census ACS socioeconomic data

## Notebook objectives

This notebook:

1. Loads and profiles all three raw datasets.
2. Standardizes schemas, datatypes, county names, and five-digit county FIPS codes.
3. evaluates missing values and exact duplicates.
4. validates domain constraints and county-year keys.
5. aggregates NOAA event-level observations to the county-year level.
6. merges NOAA, ACS, and NFIP data with explicit match diagnostics.
7. identifies potential outliers without deleting meaningful catastrophe extremes.
8. constructs `severity_metric`, `high_severity_flag`, and supporting derived features.
9. exports the final cleaned dataset as:

```text
data/processed/catastrophe_dataset.csv
```

10. generates report-ready tables for the Interim Report:
   - dataset profile,
   - missing-value summary,
   - duplicate summary,
   - datatype transformation summary,
   - outlier summary,
   - merge summary,
   - cleaning log,
   - data dictionary,
   - final validation summary.

### Important methodological decision

Extreme losses and hazard intensities are not automatically removed or winsorized. Catastrophe models are specifically concerned with tail risk, so statistically unusual observations are retained and flagged for sensitivity analysis.


## 1. Environment and reproducibility

In [ ]:
from pathlib import Path
import sys
import platform
import importlib.metadata as importlib_metadata
import pandas as pd
import numpy as np

# Resolve project root whether notebook is run from the repository root or notebooks folder.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCUMENTATION_DIR = ROOT / "documentation"
REPORT_TABLE_DIR = ROOT / "report_ready_tables"

for directory in [PROCESSED_DIR, DOCUMENTATION_DIR, REPORT_TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NOAA_FILE = RAW_DIR / "NOAA_Storm_Events_County_Year.csv"
ACS_FILE = RAW_DIR / "Census_ACS_Socioeconomic_County_Year.csv"
NFIP_FILE = RAW_DIR / "NFIP_Flood_Claims_County_Year.csv"

required_files = [NOAA_FILE, ACS_FILE, NFIP_FILE]
missing_files = [str(path.resolve()) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "The following required input files were not found:\n" + "\n".join(missing_files)
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

software_versions = pd.DataFrame({
    "Component": ["Python", "Operating System", "pandas", "numpy", "nbformat"],
    "Version": [
        platform.python_version(),
        platform.platform(),
        pd.__version__,
        np.__version__,
        importlib_metadata.version("nbformat"),
    ]
})
software_versions


## 2. Load raw datasets

In [ ]:
noaa_raw = pd.read_csv(NOAA_FILE)
acs_raw = pd.read_csv(ACS_FILE)
nfip_raw = pd.read_csv(NFIP_FILE)

raw_datasets = {
    "NOAA Storm Events": noaa_raw,
    "Census ACS": acs_raw,
    "FEMA/NFIP": nfip_raw,
}

for dataset_name, dataframe in raw_datasets.items():
    print(f"{dataset_name}: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns")
    display(dataframe.head())


## 3. Initial dataset profile

In [ ]:
def dataset_profile(name, df):
    year_col = next((c for c in df.columns if c.strip().lower() == "year"), None)
    fips_col = next((c for c in df.columns if c.strip().lower() == "county_fips"), None)
    return {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Exact Duplicate Rows": int(df.duplicated().sum()),
        "Minimum Year": int(df[year_col].min()) if year_col else None,
        "Maximum Year": int(df[year_col].max()) if year_col else None,
        "Unique Counties": int(df[fips_col].nunique()) if fips_col else None,
    }

dataset_profile_table = pd.DataFrame([
    dataset_profile(name, df) for name, df in raw_datasets.items()
])

dataset_profile_table.to_csv(REPORT_TABLE_DIR / "01_dataset_profile.csv", index=False)
dataset_profile_table


## 4. Reusable cleaning and audit functions

In [ ]:
cleaning_log = []
datatype_changes = []

def add_cleaning_log(
    step, dataset, issue, variables, detection_method, treatment,
    rows_before, rows_after, records_affected, rationale
):
    cleaning_log.append({
        "Step": step,
        "Dataset": dataset,
        "Issue": issue,
        "Variables Affected": variables,
        "Detection Method": detection_method,
        "Treatment Applied": treatment,
        "Rows Before": int(rows_before),
        "Rows After": int(rows_after),
        "Records/Values Affected": int(records_affected),
        "Rationale / Business Relevance": rationale,
    })

def standardize_text(series):
    return (
        series.astype("string")
              .str.strip()
              .str.replace(r"\s+", " ", regex=True)
    )

def minmax(series):
    numeric = pd.to_numeric(series, errors="coerce")
    minimum, maximum = numeric.min(), numeric.max()
    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(0.0, index=numeric.index)
    return (numeric - minimum) / (maximum - minimum)

def zscore(series):
    numeric = pd.to_numeric(series, errors="coerce")
    standard_deviation = numeric.std(ddof=0)
    if pd.isna(standard_deviation) or standard_deviation == 0:
        return pd.Series(0.0, index=numeric.index)
    return (numeric - numeric.mean()) / standard_deviation

def capture_datatype_changes(dataset_name, original_df, cleaned_df):
    original_lookup = {
        str(column).strip().lower().replace(" ", "_"): str(dtype)
        for column, dtype in original_df.dtypes.items()
    }
    for column in cleaned_df.columns:
        datatype_changes.append({
            "Dataset": dataset_name,
            "Variable": column,
            "Original Data Type": original_lookup.get(column, "Renamed/Derived"),
            "Final Data Type": str(cleaned_df[column].dtype),
        })


## 5. Standardize column names, text, datatypes, county names, and FIPS codes

In [ ]:
standardized_datasets = {}

for dataset_name, raw_df in [
    ("NOAA", noaa_raw),
    ("ACS", acs_raw),
    ("NFIP", nfip_raw),
]:
    df = raw_df.copy()
    rows_before = len(df)

    df.columns = (
        pd.Index(df.columns)
          .str.strip()
          .str.lower()
          .str.replace(r"[^a-z0-9]+", "_", regex=True)
          .str.strip("_")
    )

    for column in ["state", "county_name"]:
        if column in df.columns:
            df[column] = standardize_text(df[column])

    if "state" in df.columns:
        df["state"] = df["state"].str.upper()

    if "county_name" in df.columns:
        df["county_name"] = df["county_name"].str.title()

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["county_fips"] = (
        pd.to_numeric(df["county_fips"], errors="coerce")
          .astype("Int64")
          .astype("string")
          .str.zfill(5)
    )

    add_cleaning_log(
        step=1,
        dataset=dataset_name,
        issue="Inconsistent schema and merge-key formatting",
        variables="Column names, state, county_name, year, county_fips",
        detection_method="Schema inspection, text review, and datatype profiling",
        treatment=(
            "Standardized column names; trimmed whitespace; normalized state and county text; "
            "converted year to integer; converted county FIPS to a five-character key"
        ),
        rows_before=rows_before,
        rows_after=len(df),
        records_affected=0,
        rationale=(
            "Consistent county-year keys are necessary to avoid false nonmatches and make the "
            "integration process reproducible."
        ),
    )

    standardized_datasets[dataset_name] = df
    capture_datatype_changes(dataset_name, raw_df, df)

noaa = standardized_datasets["NOAA"]
acs = standardized_datasets["ACS"]
nfip = standardized_datasets["NFIP"]

display(noaa.head())
display(acs.head())
display(nfip.head())


## 6. Missing-value analysis

In [ ]:
missing_tables = []

for dataset_name, df in [("NOAA", noaa), ("ACS", acs), ("NFIP", nfip)]:
    summary = pd.DataFrame({
        "Dataset": dataset_name,
        "Variable": df.columns,
        "Missing Count": df.isna().sum().values,
        "Missing Percentage": (df.isna().mean().values * 100).round(2),
    })
    summary["Treatment"] = np.where(
        summary["Missing Count"].eq(0),
        "No treatment required",
        "Reviewed according to variable meaning and merge context"
    )
    missing_tables.append(summary)

    add_cleaning_log(
        step=2,
        dataset=dataset_name,
        issue="Missing values",
        variables="All variables",
        detection_method="Column-level null counts and missing percentages",
        treatment=(
            "No source-variable imputation was required because the supplied raw files "
            "contained no missing values."
        ),
        rows_before=len(df),
        rows_after=len(df),
        records_affected=int(df.isna().sum().sum()),
        rationale="Avoids unnecessary imputation and preserves supplied observations.",
    )

missing_value_summary = pd.concat(missing_tables, ignore_index=True)
missing_value_summary.to_csv(
    REPORT_TABLE_DIR / "02_missing_value_summary.csv", index=False
)
missing_value_summary


## 7. Exact duplicate analysis

In [ ]:
duplicate_rows = []

for dataset_name, df in [("NOAA", noaa), ("ACS", acs), ("NFIP", nfip)]:
    duplicate_count = int(df.duplicated().sum())
    rows_before = len(df)
    df.drop_duplicates(inplace=True)
    rows_after = len(df)

    duplicate_rows.append({
        "Dataset": dataset_name,
        "Rows Before": rows_before,
        "Exact Duplicate Rows": duplicate_count,
        "Rows Removed": rows_before - rows_after,
        "Rows After": rows_after,
    })

    add_cleaning_log(
        step=3,
        dataset=dataset_name,
        issue="Exact duplicate rows",
        variables="All variables",
        detection_method="pandas duplicated() across all columns",
        treatment="Removed exact duplicates",
        rows_before=rows_before,
        rows_after=rows_after,
        records_affected=duplicate_count,
        rationale="Prevents duplicate events, claims, or socioeconomic records from being counted twice.",
    )

duplicate_summary = pd.DataFrame(duplicate_rows)
duplicate_summary.to_csv(REPORT_TABLE_DIR / "03_duplicate_summary.csv", index=False)
duplicate_summary


## 8. Merge-key and domain validation

In [ ]:
for dataset_name, df in [("NOAA", noaa), ("ACS", acs), ("NFIP", nfip)]:
    invalid_key_count = int(df["year"].isna().sum() + df["county_fips"].isna().sum())
    rows_before = len(df)

    df.dropna(subset=["year", "county_fips"], inplace=True)
    df["year"] = df["year"].astype(int)

    add_cleaning_log(
        step=4,
        dataset=dataset_name,
        issue="Invalid county-year merge keys",
        variables="year, county_fips",
        detection_method="Null, datatype, and FIPS-length validation",
        treatment="Removed observations with unusable keys; none were found",
        rows_before=rows_before,
        rows_after=len(df),
        records_affected=invalid_key_count,
        rationale="County and year are mandatory fields for integrating the three data sources.",
    )

domain_rules = {
    "NOAA": {
        "wind_speed_mph": (0, None),
        "hail_size_inches": (0, None),
        "flood_depth_feet": (0, None),
        "storm_duration_hours": (0, None),
        "property_damage_usd": (0, None),
    },
    "ACS": {
        "median_income_usd": (0, None),
        "population_density_per_sq_mi": (0, None),
        "housing_age_median_years": (0, None),
        "vacancy_rate_pct": (0, 100),
        "homeownership_rate_pct": (0, 100),
        "pct_units_pre1980": (0, 100),
    },
    "NFIP": {
        "nfip_claim_count": (0, None),
        "avg_building_payout_usd": (0, None),
        "avg_contents_payout_usd": (0, None),
        "total_nfip_payout_usd": (0, None),
    },
}

domain_validation_rows = []

for dataset_name, df in [("NOAA", noaa), ("ACS", acs), ("NFIP", nfip)]:
    invalid_row_mask = pd.Series(False, index=df.index)

    for variable, (lower_bound, upper_bound) in domain_rules[dataset_name].items():
        variable_invalid = df[variable].lt(lower_bound)
        if upper_bound is not None:
            variable_invalid |= df[variable].gt(upper_bound)

        domain_validation_rows.append({
            "Dataset": dataset_name,
            "Variable": variable,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound if upper_bound is not None else "No fixed maximum",
            "Invalid Values": int(variable_invalid.sum()),
        })
        invalid_row_mask |= variable_invalid

    rows_before = len(df)
    invalid_rows = int(invalid_row_mask.sum())
    df.drop(index=df.index[invalid_row_mask], inplace=True)

    add_cleaning_log(
        step=5,
        dataset=dataset_name,
        issue="Values outside valid business/domain ranges",
        variables=", ".join(domain_rules[dataset_name].keys()),
        detection_method="Nonnegative constraints and percentage range checks",
        treatment="Removed rows violating domain rules; none were found",
        rows_before=rows_before,
        rows_after=len(df),
        records_affected=invalid_rows,
        rationale=(
            "Negative hazard/loss values and percentages outside 0–100 are not analytically valid."
        ),
    )

domain_validation_summary = pd.DataFrame(domain_validation_rows)
domain_validation_summary.to_csv(
    REPORT_TABLE_DIR / "04_domain_validation_summary.csv", index=False
)
domain_validation_summary


## 9. Aggregate NOAA event records to county-year

The NOAA file contains more than one storm event for some county-years. Since ACS and NFIP are structured at the county-year level, NOAA is aggregated as follows:

| Measure | Aggregation |
|---|---|
| Event count | Number of unique NOAA event IDs |
| Event-type count | Number of distinct event types |
| Dominant event type | Most frequent event type |
| Wind speed | Maximum |
| Hail size | Maximum |
| Flood depth | Maximum |
| Storm duration | Sum |
| Property damage | Sum |

This preserves severe hazard measurements while creating one analysis record per county-year.


In [ ]:
noaa_rows_before = len(noaa)

noaa_county_year = (
    noaa.groupby(
        ["year", "state", "county_fips", "county_name"],
        as_index=False
    )
    .agg(
        hazard_event_count=("event_id", "nunique"),
        event_type_count=("event_type", "nunique"),
        dominant_event_type=(
            "event_type",
            lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"
        ),
        wind_speed_mph=("wind_speed_mph", "max"),
        hail_size_inches=("hail_size_inches", "max"),
        flood_depth_feet=("flood_depth_feet", "max"),
        storm_duration_hours=("storm_duration_hours", "sum"),
        property_damage_usd=("property_damage_usd", "sum"),
    )
)

add_cleaning_log(
    step=6,
    dataset="NOAA",
    issue="Multiple NOAA events within the same county-year",
    variables="event_id and hazard variables",
    detection_method="Duplicate county_fips-year key analysis",
    treatment=(
        "Aggregated event counts and types; retained maximum hazard intensities; "
        "summed storm duration and property damage"
    ),
    rows_before=noaa_rows_before,
    rows_after=len(noaa_county_year),
    records_affected=noaa_rows_before - len(noaa_county_year),
    rationale=(
        "The county-year unit prevents ACS and NFIP values from being duplicated when records are merged."
    ),
)

print(f"NOAA event rows: {noaa_rows_before:,}")
print(f"NOAA county-year rows: {len(noaa_county_year):,}")
noaa_county_year.head()


## 10. Validate ACS and NFIP county-year uniqueness

In [ ]:
county_year_key_summary = []

for dataset_name, df in [("ACS", acs), ("NFIP", nfip)]:
    duplicate_key_count = int(df.duplicated(["year", "county_fips"]).sum())

    county_year_key_summary.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Duplicate County-Year Keys": duplicate_key_count,
        "Unique County-Year Keys": df[["year", "county_fips"]].drop_duplicates().shape[0],
    })

    if duplicate_key_count > 0:
        raise ValueError(
            f"{dataset_name} contains duplicate county-year keys and requires aggregation."
        )

    add_cleaning_log(
        step=7,
        dataset=dataset_name,
        issue="Duplicate county-year keys",
        variables="year, county_fips",
        detection_method="duplicated() on the composite county-year key",
        treatment="No aggregation required because the composite key was unique",
        rows_before=len(df),
        rows_after=len(df),
        records_affected=duplicate_key_count,
        rationale="A one-row-per-county-year structure is required before integration.",
    )

county_year_key_summary = pd.DataFrame(county_year_key_summary)
county_year_key_summary.to_csv(
    REPORT_TABLE_DIR / "05_county_year_key_summary.csv", index=False
)
county_year_key_summary


## 11. Merge NOAA, ACS, and NFIP with match diagnostics

In [ ]:
# NOAA is the analytic base because the study focuses on observed catastrophe events.
merged = noaa_county_year.merge(
    acs.drop(columns=["state", "county_name"]),
    on=["year", "county_fips"],
    how="left",
    validate="many_to_one",
    indicator="_acs_merge",
)

acs_match_counts = merged["_acs_merge"].value_counts().to_dict()
acs_unmatched = int((merged["_acs_merge"] != "both").sum())
merged.drop(columns="_acs_merge", inplace=True)

add_cleaning_log(
    step=8,
    dataset="Merged",
    issue="NOAA-to-ACS county-year alignment",
    variables="year, county_fips",
    detection_method="Validated left join and merge indicator",
    treatment="Retained NOAA county-years and attached matching ACS variables",
    rows_before=len(noaa_county_year),
    rows_after=len(merged),
    records_affected=acs_unmatched,
    rationale="Preserves every observed catastrophe county-year while adding socioeconomic context.",
)

merged = merged.merge(
    nfip.drop(columns=["state", "county_name"]),
    on=["year", "county_fips"],
    how="left",
    validate="many_to_one",
    indicator="_nfip_merge",
)

nfip_match_counts = merged["_nfip_merge"].value_counts().to_dict()
nfip_unmatched = int((merged["_nfip_merge"] != "both").sum())
merged["nfip_record_available"] = (merged["_nfip_merge"] == "both").astype(int)
merged.drop(columns="_nfip_merge", inplace=True)

nfip_variables = [
    "nfip_claim_count",
    "avg_building_payout_usd",
    "avg_contents_payout_usd",
    "total_nfip_payout_usd",
]
merged[nfip_variables] = merged[nfip_variables].fillna(0)

add_cleaning_log(
    step=9,
    dataset="Merged",
    issue="NOAA county-year without a matching NFIP row",
    variables=", ".join(nfip_variables),
    detection_method="Validated left join, merge indicator, and post-merge null review",
    treatment=(
        "Retained the NOAA county-year, added nfip_record_available, and assigned zero "
        "to NFIP fields for the unmatched record"
    ),
    rows_before=len(noaa_county_year),
    rows_after=len(merged),
    records_affected=nfip_unmatched,
    rationale=(
        "The availability flag preserves transparency. The valid NOAA event is not deleted, "
        "and downstream calculations remain reproducible."
    ),
)

merge_summary = pd.DataFrame([
    {
        "Merge": "NOAA → ACS",
        "NOAA Base Rows": len(noaa_county_year),
        "Matched Rows": int(acs_match_counts.get("both", 0)),
        "Unmatched NOAA Rows": acs_unmatched,
        "Match Rate Percentage": round(
            100 * int(acs_match_counts.get("both", 0)) / len(noaa_county_year), 2
        ),
    },
    {
        "Merge": "NOAA/ACS → NFIP",
        "NOAA Base Rows": len(noaa_county_year),
        "Matched Rows": int(nfip_match_counts.get("both", 0)),
        "Unmatched NOAA Rows": nfip_unmatched,
        "Match Rate Percentage": round(
            100 * int(nfip_match_counts.get("both", 0)) / len(noaa_county_year), 2
        ),
    },
])

merge_summary.to_csv(REPORT_TABLE_DIR / "06_merge_summary.csv", index=False)
merge_summary


## 12. Outlier identification and treatment

In [ ]:
outlier_variables = [
    "wind_speed_mph",
    "hail_size_inches",
    "flood_depth_feet",
    "storm_duration_hours",
    "property_damage_usd",
    "nfip_claim_count",
    "total_nfip_payout_usd",
    "median_income_usd",
    "population_density_per_sq_mi",
]

outlier_rows = []

for variable in outlier_variables:
    q1 = merged[variable].quantile(0.25)
    q3 = merged[variable].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    flag_variable = f"{variable}_outlier_flag"
    merged[flag_variable] = (
        (merged[variable] < lower_fence) |
        (merged[variable] > upper_fence)
    ).astype(int)

    outlier_rows.append({
        "Variable": variable,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Fence": lower_fence,
        "Upper Fence": upper_fence,
        "Outlier Count": int(merged[flag_variable].sum()),
        "Outlier Percentage": round(100 * merged[flag_variable].mean(), 2),
        "Treatment": "Retained and flagged",
    })

outlier_summary = pd.DataFrame(outlier_rows)

add_cleaning_log(
    step=10,
    dataset="Merged",
    issue="Potential statistical outliers",
    variables=", ".join(outlier_variables),
    detection_method="1.5 × IQR rule",
    treatment="Retained all observations and created variable-specific outlier flags",
    rows_before=len(merged),
    rows_after=len(merged),
    records_affected=int(outlier_summary["Outlier Count"].sum()),
    rationale=(
        "Extreme catastrophe observations may represent genuine tail risk. Removing them could "
        "understate loss severity and weaken the business relevance of the analysis."
    ),
)

outlier_summary.to_csv(REPORT_TABLE_DIR / "07_outlier_summary.csv", index=False)
outlier_summary


## 13. Feature engineering

### Loss variables

- `total_observed_loss_usd` combines NOAA property damage and NFIP payout.
- Natural-log transformations reduce strong right skew while retaining rank information.

### Severity metric

`severity_metric` is the average of six min-max normalized components:

1. log NOAA property damage,
2. log NFIP payout,
3. wind speed,
4. hail size,
5. flood depth,
6. storm duration.

The metric ranges from 0 to 1. It provides one transparent county-year severity measure that combines physical intensity and financial impact.

### High-severity target

`high_severity_flag = 1` for county-years at or above the 80th percentile of `severity_metric`. This creates a clearly defined binary target for RQ3 classification modeling.

### Socioeconomic resilience index

The index averages standardized income and homeownership contributions and subtracts standardized housing age, vacancy, and pre-1980 housing share. It is retained as an exploratory/moderation feature rather than treated as an externally validated index.


In [ ]:
merged["total_observed_loss_usd"] = (
    merged["property_damage_usd"] + merged["total_nfip_payout_usd"]
)

merged["log_property_damage"] = np.log1p(merged["property_damage_usd"])
merged["log_total_nfip_payout"] = np.log1p(merged["total_nfip_payout_usd"])
merged["log_total_observed_loss"] = np.log1p(merged["total_observed_loss_usd"])

severity_components = pd.DataFrame({
    "log_property_damage": minmax(merged["log_property_damage"]),
    "log_total_nfip_payout": minmax(merged["log_total_nfip_payout"]),
    "wind_speed_mph": minmax(merged["wind_speed_mph"]),
    "hail_size_inches": minmax(merged["hail_size_inches"]),
    "flood_depth_feet": minmax(merged["flood_depth_feet"]),
    "storm_duration_hours": minmax(merged["storm_duration_hours"]),
})

merged["severity_metric"] = severity_components.mean(axis=1)

severity_threshold = merged["severity_metric"].quantile(0.80)
merged["high_severity_flag"] = (
    merged["severity_metric"] >= severity_threshold
).astype(int)

merged["socioeconomic_resilience_index"] = (
    zscore(merged["median_income_usd"])
    + zscore(merged["homeownership_rate_pct"])
    - zscore(merged["housing_age_median_years"])
    - zscore(merged["vacancy_rate_pct"])
    - zscore(merged["pct_units_pre1980"])
) / 5

feature_engineering_summary = pd.DataFrame([
    {
        "Derived Variable": "total_observed_loss_usd",
        "Formula / Rule": "property_damage_usd + total_nfip_payout_usd",
        "Purpose": "Combined financial-impact measure",
    },
    {
        "Derived Variable": "log_property_damage",
        "Formula / Rule": "ln(1 + property_damage_usd)",
        "Purpose": "Reduce right skew for modeling",
    },
    {
        "Derived Variable": "log_total_nfip_payout",
        "Formula / Rule": "ln(1 + total_nfip_payout_usd)",
        "Purpose": "Reduce right skew for modeling",
    },
    {
        "Derived Variable": "log_total_observed_loss",
        "Formula / Rule": "ln(1 + total_observed_loss_usd)",
        "Purpose": "Combined transformed loss measure",
    },
    {
        "Derived Variable": "severity_metric",
        "Formula / Rule": "Mean of six min-max normalized hazard/loss components",
        "Purpose": "Continuous severity target for RQ1, RQ2, and RQ4",
    },
    {
        "Derived Variable": "high_severity_flag",
        "Formula / Rule": f"1 when severity_metric ≥ 80th percentile ({severity_threshold:.6f})",
        "Purpose": "Binary classification target for RQ3",
    },
    {
        "Derived Variable": "socioeconomic_resilience_index",
        "Formula / Rule": "Average of standardized protective factors minus standardized vulnerability factors",
        "Purpose": "Exploratory socioeconomic moderator",
    },
])

add_cleaning_log(
    step=11,
    dataset="Merged",
    issue="Analysis variables required by RQ1–RQ4 were not directly available",
    variables=", ".join(feature_engineering_summary["Derived Variable"]),
    detection_method="Research-question and model-input review",
    treatment="Created transparent loss, severity, classification, and resilience features",
    rows_before=len(merged),
    rows_after=len(merged),
    records_affected=len(merged),
    rationale=(
        "The engineered variables provide consistent targets and predictors for regression, "
        "moderation, classification, and trend analysis."
    ),
)

feature_engineering_summary.to_csv(
    REPORT_TABLE_DIR / "08_feature_engineering_summary.csv", index=False
)
feature_engineering_summary


## 14. Final dataset ordering and validation

In [ ]:
core_variables = [
    "year",
    "state",
    "county_fips",
    "county_name",
    "hazard_event_count",
    "event_type_count",
    "dominant_event_type",
    "wind_speed_mph",
    "hail_size_inches",
    "flood_depth_feet",
    "storm_duration_hours",
    "property_damage_usd",
    "nfip_claim_count",
    "avg_building_payout_usd",
    "avg_contents_payout_usd",
    "total_nfip_payout_usd",
    "nfip_record_available",
    "median_income_usd",
    "population_density_per_sq_mi",
    "housing_age_median_years",
    "construction_type",
    "vacancy_rate_pct",
    "homeownership_rate_pct",
    "pct_units_pre1980",
    "total_observed_loss_usd",
    "log_property_damage",
    "log_total_nfip_payout",
    "log_total_observed_loss",
    "severity_metric",
    "high_severity_flag",
    "socioeconomic_resilience_index",
]

other_variables = [
    variable for variable in merged.columns
    if variable not in core_variables
]

merged = (
    merged[core_variables + other_variables]
    .sort_values(["year", "county_fips"])
    .reset_index(drop=True)
)

validation_checks = [
    {
        "Validation Check": "No duplicate county-year keys",
        "Result": int(merged.duplicated(["year", "county_fips"]).sum()),
        "Expected": 0,
        "Status": "Pass" if merged.duplicated(["year", "county_fips"]).sum() == 0 else "Fail",
    },
    {
        "Validation Check": "No missing values in final dataset",
        "Result": int(merged.isna().sum().sum()),
        "Expected": 0,
        "Status": "Pass" if merged.isna().sum().sum() == 0 else "Fail",
    },
    {
        "Validation Check": "All FIPS codes contain five characters",
        "Result": int(merged["county_fips"].str.len().eq(5).sum()),
        "Expected": len(merged),
        "Status": "Pass" if merged["county_fips"].str.len().eq(5).all() else "Fail",
    },
    {
        "Validation Check": "Severity metric is between 0 and 1",
        "Result": int(merged["severity_metric"].between(0, 1).sum()),
        "Expected": len(merged),
        "Status": "Pass" if merged["severity_metric"].between(0, 1).all() else "Fail",
    },
    {
        "Validation Check": "High-severity flag contains only 0 or 1",
        "Result": sorted(merged["high_severity_flag"].unique().tolist()),
        "Expected": [0, 1],
        "Status": (
            "Pass"
            if set(merged["high_severity_flag"].unique()).issubset({0, 1})
            else "Fail"
        ),
    },
]

final_validation_summary = pd.DataFrame(validation_checks)

if not final_validation_summary["Status"].eq("Pass").all():
    display(final_validation_summary)
    raise ValueError("One or more final validation checks failed.")

add_cleaning_log(
    step=12,
    dataset="Merged",
    issue="Final analysis-readiness validation",
    variables="All fields and county-year key",
    detection_method="Programmatic assertions and validation summary",
    treatment="Confirmed uniqueness, completeness, FIPS format, and target-variable validity",
    rows_before=len(merged),
    rows_after=len(merged),
    records_affected=0,
    rationale="Ensures that the exported county-year dataset is reproducible and ready for EDA.",
)

final_validation_summary.to_csv(
    REPORT_TABLE_DIR / "09_final_validation_summary.csv", index=False
)
final_validation_summary


## 15. Datatype transformation summary

In [ ]:
datatype_transformation_summary = pd.DataFrame(datatype_changes)
datatype_transformation_summary.to_csv(
    REPORT_TABLE_DIR / "10_datatype_transformation_summary.csv", index=False
)
datatype_transformation_summary


## 16. Build the data cleaning log

In [ ]:
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(
    DOCUMENTATION_DIR / "data_cleaning_log_v2.csv", index=False
)
cleaning_log_df.to_csv(
    REPORT_TABLE_DIR / "11_data_cleaning_log.csv", index=False
)
cleaning_log_df


## 17. Build the final data dictionary

In [ ]:

variable_definitions = {
    "year": ("Calendar year of the hazard event and matched county-year observations", "All", "RQ1–RQ4"),
    "state": ("Two-letter U.S. state abbreviation", "NOAA", "Grouping and descriptive analysis"),
    "county_fips": ("Five-digit county Federal Information Processing Standards code", "All", "Primary merge key"),
    "county_name": ("County name", "NOAA", "Grouping and descriptive analysis"),
    "hazard_event_count": ("Number of unique NOAA hazard events in the county-year", "Engineered from NOAA", "Exposure/control"),
    "event_type_count": ("Number of distinct NOAA event types in the county-year", "Engineered from NOAA", "Exposure/control"),
    "dominant_event_type": ("Most frequently occurring NOAA event type in the county-year", "Engineered from NOAA", "EDA and segmentation"),
    "wind_speed_mph": ("Maximum recorded wind speed in the county-year", "NOAA", "RQ1, RQ2, and RQ3"),
    "hail_size_inches": ("Maximum recorded hail diameter in the county-year", "NOAA", "RQ1, RQ2, and RQ3"),
    "flood_depth_feet": ("Maximum recorded flood depth in the county-year", "NOAA", "RQ1, RQ2, and RQ3"),
    "storm_duration_hours": ("Total duration of NOAA events in the county-year", "Engineered from NOAA", "RQ1, RQ2, and RQ3"),
    "property_damage_usd": ("Total NOAA-reported property damage in the county-year", "Engineered from NOAA", "Severity construction"),
    "nfip_claim_count": ("Number of NFIP claims in the county-year", "NFIP", "Exposure and descriptive analysis"),
    "avg_building_payout_usd": ("Average NFIP building payout", "NFIP", "EDA and controls"),
    "avg_contents_payout_usd": ("Average NFIP contents payout", "NFIP", "EDA and controls"),
    "total_nfip_payout_usd": ("Total NFIP payout in the county-year", "NFIP", "Severity construction"),
    "nfip_record_available": ("1 if an NFIP record matched the county-year; otherwise 0", "Engineered", "Merge-quality indicator"),
    "median_income_usd": ("Median household income", "Census ACS", "RQ2 and RQ3"),
    "population_density_per_sq_mi": ("Population per square mile", "Census ACS", "RQ2 and RQ3"),
    "housing_age_median_years": ("Median age of housing units", "Census ACS", "RQ2 and RQ3"),
    "construction_type": ("Predominant construction category in the supplied ACS dataset", "Census ACS", "RQ2 and RQ3"),
    "vacancy_rate_pct": ("Vacant housing units as a percentage", "Census ACS", "RQ2 and RQ3"),
    "homeownership_rate_pct": ("Owner-occupied housing units as a percentage", "Census ACS", "RQ2 and RQ3"),
    "pct_units_pre1980": ("Housing units constructed before 1980 as a percentage", "Census ACS", "RQ2 and RQ3"),
    "total_observed_loss_usd": ("NOAA property damage plus total NFIP payout", "Engineered", "Financial severity context"),
    "log_property_damage": ("Natural logarithm of one plus NOAA property damage", "Engineered", "Modeling transformation"),
    "log_total_nfip_payout": ("Natural logarithm of one plus total NFIP payout", "Engineered", "Modeling transformation"),
    "log_total_observed_loss": ("Natural logarithm of one plus total observed loss", "Engineered", "Modeling transformation"),
    "severity_metric": ("Mean of six min-max normalized hazard and loss components", "Engineered", "Continuous target for RQ1, RQ2, and RQ4"),
    "high_severity_flag": ("1 for observations at or above the 80th percentile of severity_metric", "Engineered", "Binary target for RQ3"),
    "socioeconomic_resilience_index": ("Exploratory standardized index combining income, homeownership, housing age, vacancy, and pre-1980 housing share", "Engineered", "RQ2 and exploratory EDA"),
}

dictionary_rows = []

for variable in merged.columns:
    if variable.endswith("_outlier_flag"):
        base_variable = variable.removesuffix("_outlier_flag")
        definition = (
            f"Indicator equal to 1 when {base_variable} falls outside the 1.5 × IQR fences"
        )
        source = "Engineered"
        intended_use = "Sensitivity analysis"
        notes = "Extreme observation retained; flag supports sensitivity checks"
    else:
        definition, source, intended_use = variable_definitions.get(
            variable,
            ("Analysis variable", "Derived", "Analysis")
        )
        notes = ""

    series = merged[variable]

    if pd.api.types.is_integer_dtype(series):
        data_type = "Integer"
    elif pd.api.types.is_float_dtype(series):
        data_type = "Numeric (decimal)"
    else:
        data_type = "Categorical/Text"

    if variable == "county_fips":
        observed_range = "Five-character code, including leading zeros"
    elif variable.endswith("_flag") or variable == "nfip_record_available":
        observed_range = "0 or 1"
    elif variable.endswith("_pct"):
        observed_range = "0–100"
    elif pd.api.types.is_numeric_dtype(series):
        observed_range = f"{series.min():.4g} to {series.max():.4g}"
    else:
        values = sorted(series.astype(str).unique())
        observed_range = ", ".join(values[:8]) + ("…" if len(values) > 8 else "")

    dictionary_rows.append({
        "Variable Name": variable,
        "Definition": definition,
        "Data Type": data_type,
        "Allowed Values / Observed Range": observed_range,
        "Source": source,
        "Missing Values Handling": "No missing values in the final dataset",
        "RQ / Intended Use": intended_use,
        "Notes": notes,
    })

data_dictionary_df = pd.DataFrame(dictionary_rows)
data_dictionary_df.to_csv(
    DOCUMENTATION_DIR / "data_dictionary_v2.csv", index=False
)
data_dictionary_df.to_csv(
    REPORT_TABLE_DIR / "12_data_dictionary.csv", index=False
)
data_dictionary_df


## 18. Export the cleaned dataset and final report summary

In [ ]:
cleaned_output = PROCESSED_DIR / "catastrophe_dataset.csv"
merged.to_csv(cleaned_output, index=False)

phase1_summary = pd.DataFrame([
    {"Metric": "Raw NOAA rows", "Value": len(noaa_raw)},
    {"Metric": "NOAA county-year rows after aggregation", "Value": len(noaa_county_year)},
    {"Metric": "Raw ACS rows", "Value": len(acs_raw)},
    {"Metric": "Raw NFIP rows", "Value": len(nfip_raw)},
    {"Metric": "Final cleaned rows", "Value": len(merged)},
    {"Metric": "Final columns", "Value": merged.shape[1]},
    {"Metric": "Minimum year", "Value": int(merged["year"].min())},
    {"Metric": "Maximum year", "Value": int(merged["year"].max())},
    {"Metric": "Unique counties", "Value": int(merged["county_fips"].nunique())},
    {"Metric": "Final missing values", "Value": int(merged.isna().sum().sum())},
    {
        "Metric": "Duplicate county-year keys",
        "Value": int(merged.duplicated(["year", "county_fips"]).sum())
    },
    {"Metric": "Unmatched ACS county-years", "Value": acs_unmatched},
    {"Metric": "Unmatched NFIP county-years", "Value": nfip_unmatched},
    {"Metric": "High-severity threshold", "Value": round(float(severity_threshold), 6)},
    {"Metric": "High-severity observations", "Value": int(merged["high_severity_flag"].sum())},
])

phase1_summary.to_csv(
    DOCUMENTATION_DIR / "phase1_quality_summary_v2.csv", index=False
)
phase1_summary.to_csv(
    REPORT_TABLE_DIR / "13_phase1_quality_summary.csv", index=False
)

print(f"Cleaned dataset saved to: {cleaned_output.resolve()}")
print(f"Final dataset shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns")
display(phase1_summary)
display(merged.head())


## 19. Interim Report-ready interpretation

The cleaning process produced a single county-year dataset integrating hazard, insurance-loss, and socioeconomic information. NOAA event-level observations were aggregated to the county-year level to align with ACS and NFIP records. Column names, text fields, years, and county FIPS codes were standardized before integration. No source-level missing values or exact duplicate rows required corrective treatment.

All NOAA county-year observations matched ACS records. One NOAA county-year did not match an NFIP record. That observation was retained because it represented a valid hazard event; an explicit `nfip_record_available` indicator was added, and NFIP measures were set to zero for that unmatched county-year. This treatment is documented in the cleaning log and can be evaluated in later sensitivity analysis.

Potential outliers were identified with the 1.5 × IQR rule. They were not deleted because unusually severe hazard and loss observations are central to catastrophe-risk analysis. Instead, variable-specific outlier flags were added.

The final dataset contains one observation per county-year, has no remaining missing values, has no duplicate county-year keys, and includes engineered continuous and binary severity targets required for the four research questions.
